In [32]:
from pyscf import gto, scf, dft, lib
import numpy as np
np.set_printoptions(10, linewidth=150)

In [33]:
mol = gto.Mole(atom="O; H 1 0.94; H 1 0.94 2 104.5", basis="def2-TZVP").build()

In [34]:
mf = scf.RKS(mol, xc="TPSS0").run()

converged SCF energy = -76.4430874743456


In [35]:
arrays = {
    "mo_coeff": np.asarray(mf.mo_coeff, order="C"),
    "mo_occ": np.asarray(mf.mo_occ, order="C"),
    "rdm1": mf.make_rdm1(),
    "coords": mf.grids.coords,
    "weights": mf.grids.weights,
}

In [36]:
mo_coeff = np.asarray(mf.mo_coeff, order="C")
mo_occ = np.asarray(mf.mo_occ, order="C")
rdm1 = mf.make_rdm1()
coords = mf.grids.coords
weights = mf.grids.weights

In [37]:
grids = mf.grids

In [38]:
ni = dft.numint.NumInt()

In [39]:
np.savez("h2o", **arrays)

## Get rho by dm

In [40]:
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho_rho = ni.eval_rho(mol, ao[0], rdm1, xctype="lda")
rho_sigma = ni.eval_rho(mol, ao[0:4], rdm1, xctype="gga")
rho_tau = ni.eval_rho(mol, ao[0:4], rdm1, xctype="mgga", with_lapl=False)
rho_lapl = ni.eval_rho(mol, ao[0:10], rdm1, xctype="mgga", with_lapl=True)

In [41]:
rho_rho.shape, lib.fp(rho_rho)

((33704,), np.float64(-438.0303348067813))

In [42]:
rho_sigma.shape, lib.fp(rho_sigma)

((4, 33704), np.float64(25704.144800854458))

In [43]:
rho_tau.shape, lib.fp(rho_tau)

((5, 33704), np.float64(17140.300791589918))

In [44]:
rho_lapl_ = rho_lapl[[0, 1, 2, 3, 5, 4]]

In [45]:
rho_lapl_.shape, lib.fp(rho_lapl_[0:5]), lib.fp(rho_lapl_[5])

((6, 33704), np.float64(17140.300791589918), np.float64(2470300.1875723414))

## xc_eff

### RHO (LDA)

In [46]:
result = ni.eval_xc_eff("LDA_X", rho_rho, deriv=3)

In [47]:
[r.shape for r in result]

[(33704,), (1, 33704), (1, 1, 33704), (1, 1, 1, 33704)]

In [48]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_rho * weights),
    lib.fp(result[3] * rho_rho * rho_rho * weights),
])
fps

array([-0.0653646142, -0.0871528189, -0.0290509396,  0.0193672931])

### SIGMA (GGA)

In [49]:
result = ni.eval_xc_eff("GGA_X_PBE", rho_sigma, deriv=3)

In [50]:
[r.shape for r in result]

[(33704,), (4, 33704), (4, 4, 33704), (4, 4, 4, 33704)]

In [51]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_sigma * weights),
    lib.fp(result[3] * rho_sigma * rho_sigma[:, None, :] * weights),
])
fps

array([-0.1652985961, -0.2296325511, -0.1386410873, -0.523551635 ])

### TAU (MGGA)

In [52]:
result = ni.eval_xc_eff("HYB_MGGA_XC_TPSSH", rho_tau, deriv=3)

In [53]:
[r.shape for r in result]

[(33704,), (5, 33704), (5, 5, 33704), (5, 5, 5, 33704)]

In [54]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_tau * weights),
    lib.fp(result[3] * rho_tau * rho_tau[:, None, :] * weights),
])
fps

array([-0.1378400498, -0.1893572125, -0.1183454954, -1.0447740366])

In [55]:
%%time
result = ni.eval_xc_eff("HYB_MGGA_XC_TPSSH", rho_tau, deriv=3)

CPU times: user 1.4 s, sys: 38 ms, total: 1.44 s
Wall time: 56.9 ms


## exc, vxc, fxc, kxc

In [56]:
# Setting up (density)
dm0 = rdm1
dm1 = (mol.intor("int1e_r") + mol.intor("int1e_giao_irjxp")) * rdm1
dm2 = mol.intor("int1e_rr") * rdm1

### RHO (LDA)

In [57]:
# Setting up (density grids and eff potentials)
(_, ngrids, nao) = ao.shape
rho0 = ni.eval_rho(mol, ao[0], dm0, xctype="lda")
rho1 = np.array([ni.eval_rho(mol, ao[0], d, xctype="lda") for d in dm1]).reshape([len(dm1), -1, ngrids])
rho2 = np.array([ni.eval_rho(mol, ao[0], d, xctype="lda") for d in dm2]).reshape([len(dm2), -1, ngrids])
exc_eff, vxc_eff, fxc_eff, kxc_eff = ni.eval_xc_eff("LDA_X", rho0, deriv=3)

In [58]:
# canonical way of exc/vxc
nelec, exc, vxc = ni.nr_rks(mol, grids, "LDA_X", dm0)
np.array([
    exc,
    lib.fp(vxc)
])

array([ -8.1384975323, -27.2331156537])

In [59]:
# naive blas-3 way of exc/vxc
exc_recap = (exc_eff * rho0 * weights).sum()
vxc_recap = ao[0].T * (weights * vxc_eff[0]) @ ao[0]
assert np.allclose(exc_recap, exc)
assert np.allclose(vxc_recap, vxc)

In [60]:
# canonical way of fxc
fxc = ni.nr_rks_fxc(mol, grids, "LDA_X", dm0, dm1)
fxc.shape, lib.fp(fxc)

((3, 43, 43), np.float64(-0.09693300035134958))

In [61]:
# naive blas-3 way of fxc
fxc_contract = (fxc_eff[None, :, :, :] * rho1[:, None, :, :]).sum(axis=-2) * weights
#                        [x] v0 v1  g         x  [v0] v1  g
fxc_recap = np.zeros([len(dm1), nao, nao])
for x in range(len(dm1)):
    fxc_recap[x] = ao[0].T * fxc_contract[x, 0] @ ao[0]
assert np.allclose(fxc_recap, fxc)

In [62]:
# naive blas-3 way of kxc
kxc_contract = (kxc_eff[None, None, :, :, :, :] * rho1[None, :, None, None, :, :] * rho2[:, None, None, :, None, :]).sum(axis=(-2, -3)) * weights
#                        [y]   [x] v0 v2 v1  g          [y]  x  [v0]  [v2] v1  g         y   [x]  [v0] v2  [v1]  g
kxc_recap = np.zeros([len(dm2), len(dm1), nao, nao])
for y in range(len(dm2)):
    for x in range(len(dm1)):
        kxc_recap[y, x] = ao[0].T * kxc_contract[y, x, 0] @ ao[0]
kxc_recap.shape, lib.fp(kxc_recap)

((9, 3, 43, 43), np.float64(0.37891650918264397))

### SIGMA (GGA)

In [63]:
# Setting up (density grids and eff potentials)
(_, ngrids, nao) = ao.shape
rho0 = ni.eval_rho(mol, ao[:4], dm0, xctype="gga")
rho1 = np.array([ni.eval_rho(mol, ao[:4], d, xctype="gga") for d in dm1]).reshape([len(dm1), -1, ngrids])
rho2 = np.array([ni.eval_rho(mol, ao[:4], d, xctype="gga") for d in dm2]).reshape([len(dm2), -1, ngrids])
exc_eff, vxc_eff, fxc_eff, kxc_eff = ni.eval_xc_eff("GGA_X_PBE", rho0, deriv=3)

In [64]:
# canonical way of exc/vxc
nelec, exc, vxc = ni.nr_rks(mol, grids, "GGA_X_PBE", dm0)
np.array([
    exc,
    lib.fp(vxc)
])

array([ -8.9542650216, -28.6270372279])

In [65]:
# naive blas-3 way of exc/vxc
exc_recap = (exc_eff * rho0[0] * weights).sum()
vxc_recap = (
    + 0.5 * ao[0].T * (weights * vxc_eff[0]) @ ao[0]
    + ao[1].T * (weights * vxc_eff[1]) @ ao[0]
    + ao[2].T * (weights * vxc_eff[2]) @ ao[0]
    + ao[3].T * (weights * vxc_eff[3]) @ ao[0]
)
vxc_recap += vxc_recap.swapaxes(-1, -2)
assert np.allclose(exc_recap, exc)
assert np.allclose(vxc_recap, vxc)

In [66]:
# canonical way of fxc
fxc = ni.nr_rks_fxc(mol, grids, "GGA_X_PBE", dm0, dm1)
fxc.shape, lib.fp(fxc)

((3, 43, 43), np.float64(-0.10389233031802968))

In [67]:
# naive blas-3 way of fxc
fxc_contract = (fxc_eff[None, :, :, :] * rho1[:, None, :, :]).sum(axis=-2) * weights
#                        [x] v0 v1  g         x  [v0] v1  g
# shape : [len(dim1), nvar, ngrids]
fxc_recap = np.zeros([len(dm1), nao, nao])
for x in range(len(dm1)):
    fxc_recap[x] = (
        + 0.5 * ao[0].T * fxc_contract[x, 0] @ ao[0]
        + ao[1].T * fxc_contract[x, 1] @ ao[0]
        + ao[2].T * fxc_contract[x, 2] @ ao[0]
        + ao[3].T * fxc_contract[x, 3] @ ao[0]
    )
fxc_recap += fxc_recap.swapaxes(-1, -2)
assert np.allclose(fxc_recap, fxc)

In [68]:
# naive blas-3 way of kxc
kxc_contract = (kxc_eff[None, None, :, :, :, :] * rho1[None, :, None, None, :, :] * rho2[:, None, None, :, None, :]).sum(axis=(-2, -3)) * weights
#                        [y]   [x] v0 v2 v1  g          [y]  x  [v0]  [v2] v1  g         y   [x]  [v0] v2  [v1]  g
# shape : [len(dm2), len(dm1), nvar, ngrids]
kxc_recap = np.zeros([len(dm2), len(dm1), nao, nao])
for y in range(len(dm2)):
    for x in range(len(dm1)):
        kxc_recap[y, x] = (
            + 0.5 * ao[0].T * kxc_contract[y, x, 0] @ ao[0]
            + ao[1].T * kxc_contract[y, x, 1] @ ao[0]
            + ao[2].T * kxc_contract[y, x, 2] @ ao[0]
            + ao[3].T * kxc_contract[y, x, 3] @ ao[0]
        )
kxc_recap += kxc_recap.swapaxes(-1, -2)
kxc_recap.shape, lib.fp(kxc_recap)

((9, 3, 43, 43), np.float64(0.40594124509385127))

### TAU (MGGA)

In [69]:
# Setting up (density grids and eff potentials)
(_, ngrids, nao) = ao.shape
rho0 = ni.eval_rho(mol, ao[:4], dm0, xctype="mgga", with_lapl=False)
rho1 = np.array([ni.eval_rho(mol, ao[:4], d, xctype="mgga", with_lapl=False) for d in dm1]).reshape([len(dm1), -1, ngrids])
rho2 = np.array([ni.eval_rho(mol, ao[:4], d, xctype="mgga", with_lapl=False) for d in dm2]).reshape([len(dm2), -1, ngrids])
exc_eff, vxc_eff, fxc_eff, kxc_eff = ni.eval_xc_eff("HYB_MGGA_XC_TPSSH", rho0, deriv=3)

In [70]:
# canonical way of exc/vxc
nelec, exc, vxc = ni.nr_rks(mol, grids, "HYB_MGGA_XC_TPSSH", dm0)
np.array([
    exc,
    lib.fp(vxc)
])

array([ -8.4667246286, -26.3517912584])

In [71]:
# naive blas-3 way of exc/vxc
exc_recap = (exc_eff * rho0[0] * weights).sum()
vxc_recap = (
    + 0.5 * ao[0].T * (weights * vxc_eff[0]) @ ao[0]
    + ao[1].T * (weights * vxc_eff[1]) @ ao[0]
    + ao[2].T * (weights * vxc_eff[2]) @ ao[0]
    + ao[3].T * (weights * vxc_eff[3]) @ ao[0]
    + 0.25 * ao[1].T * (weights * vxc_eff[4]) @ ao[1]
    + 0.25 * ao[2].T * (weights * vxc_eff[4]) @ ao[2]
    + 0.25 * ao[3].T * (weights * vxc_eff[4]) @ ao[3]
)
vxc_recap += vxc_recap.swapaxes(-1, -2)
assert np.allclose(exc_recap, exc)
assert np.allclose(vxc_recap, vxc)

In [72]:
# canonical way of fxc
fxc = ni.nr_rks_fxc(mol, grids, "HYB_MGGA_XC_TPSSH", dm0, dm1)
fxc.shape, lib.fp(fxc)

((3, 43, 43), np.float64(-0.09110536214579404))

In [73]:
# naive blas-3 way of fxc
fxc_contract = (fxc_eff[None, :, :, :] * rho1[:, None, :, :]).sum(axis=-2) * weights
#                        [x] v0 v1  g         x  [v0] v1  g
# shape : [len(dim1), nvar, ngrids]
fxc_recap = np.zeros([len(dm1), nao, nao])
for x in range(len(dm1)):
    fxc_recap[x] = (
        + 0.5 * ao[0].T * fxc_contract[x, 0] @ ao[0]
        + ao[1].T * fxc_contract[x, 1] @ ao[0]
        + ao[2].T * fxc_contract[x, 2] @ ao[0]
        + ao[3].T * fxc_contract[x, 3] @ ao[0]
        + 0.25 * ao[1].T * fxc_contract[x, 4] @ ao[1]
        + 0.25 * ao[2].T * fxc_contract[x, 4] @ ao[2]
        + 0.25 * ao[3].T * fxc_contract[x, 4] @ ao[3]
    )
fxc_recap += fxc_recap.swapaxes(-1, -2)
assert np.allclose(fxc_recap, fxc)

In [74]:
# naive blas-3 way of kxc
kxc_contract = (kxc_eff[None, None, :, :, :, :] * rho1[None, :, None, None, :, :] * rho2[:, None, None, :, None, :]).sum(axis=(-2, -3)) * weights
#                        [y]   [x] v0 v2 v1  g          [y]  x  [v0]  [v2] v1  g         y   [x]  [v0] v2  [v1]  g
# shape : [len(dm2), len(dm1), nvar, ngrids]
kxc_recap = np.zeros([len(dm2), len(dm1), nao, nao])
for y in range(len(dm2)):
    for x in range(len(dm1)):
        kxc_recap[y, x] = (
            + 0.5 * ao[0].T * kxc_contract[y, x, 0] @ ao[0]
            + ao[1].T * kxc_contract[y, x, 1] @ ao[0]
            + ao[2].T * kxc_contract[y, x, 2] @ ao[0]
            + ao[3].T * kxc_contract[y, x, 3] @ ao[0]
            + 0.25 * ao[1].T * kxc_contract[y, x, 4] @ ao[1]
            + 0.25 * ao[2].T * kxc_contract[y, x, 4] @ ao[2]
            + 0.25 * ao[3].T * kxc_contract[y, x, 4] @ ao[3]
        )
kxc_recap += kxc_recap.swapaxes(-1, -2)
kxc_recap.shape, lib.fp(kxc_recap)

((9, 3, 43, 43), np.float64(0.5466595210063728))